In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
import pandas as pd

from heart_disease.data.ingestion import load_file
from heart_disease.data.splitting import split_data
import heart_disease.models.train as t
from heart_disease.features.preprocessing import (
    create_training_pipeline,
    create_categorical_pipeline,
    create_numeric_pipeline,
    create_preprocessor,
)
from heart_disease.config import (
    INTERIM_DATA_DIR,
    RANDOM_STATE,
    TARGET_COLUMN,
    FEATURES,
    MODEL_DIR,
    ExperimentConfig,
)
from heart_disease.models.tracking import start_experiment

# Import and Split Data

In [2]:
df = load_file(INTERIM_DATA_DIR / "cleaned_dataset.csv")
df.head()

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target
0,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,1
2,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [3]:
X_train, X_test, y_train, y_test = split_data(df, TARGET_COLUMN, FEATURES)
X_test.to_parquet(INTERIM_DATA_DIR / "X_test.parquet", index=False)
y_test.to_frame("target").to_parquet(INTERIM_DATA_DIR / "y_test.parquet", index=False)

2026-09-15 19:36:13,059 | INFO | heart_disease.data.splitting | Created with shape of X_train: (736, 13), X_test: (184, 13), y_train: (736,), y_test: (184,)


# Build Baseline Model

In [4]:
logistic_regression = create_training_pipeline(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
)

t.cross_validate_model(
    logistic_regression, X_train, y_train, t.TrainingConfig(ExperimentConfig())
)

2026-09-15 19:36:13,189 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:14,616 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.832 ± 0.011


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.816568,0.823529,0.835443,0.836364,0.849673,0.832315,0.011425


# Evaluate Dataset Based On Model Performance

## Evaluate maximum missing data allowed in each observation

Each observation contains fourteen features that may provide useful predictive information.
However, some observation have more than half of their features missing, potentially indicating poor data quality. 
Therefore, we evaluate different threshold for removing observations based on their proportion of missing values using cross-validation. 
The selected threshold must satisfy two constraint: the proportion of discarded observations must not exceed `ten percent` of the dataset, and the refined dataset must have less than one percent difference in target mean with the original dataset.

In [5]:
results = []

for num in range(6, 14):
    initial_rows = X_train.shape[0]
    temporary_df = X_train.copy()
    temporary_df[TARGET_COLUMN] = y_train.copy()
    temporary_df = temporary_df.dropna(thresh=num)

    X = temporary_df[FEATURES]
    y = temporary_df[TARGET_COLUMN]

    scores = t.cross_validate_model(
        logistic_regression, X, y, t.TrainingConfig(ExperimentConfig())
    )

    scores["threshold"] = num
    scores["total_rows"] = X.shape[0]
    scores["%_rows_deleted"] = (initial_rows - X.shape[0]) / initial_rows * 100
    scores["total_rows x mean"] = scores["total_rows"] * scores["mean"]
    scores["target_mean"] = y.mean()
    results.append(scores)

results = pd.concat(results)
results = results.sort_values("total_rows x mean", ascending=False).set_index(
    "threshold"
)
results

2026-09-15 19:36:14,784 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1


2026-09-15 19:36:15,860 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.832 ± 0.011
2026-09-15 19:36:15,873 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:17,188 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.844 ± 0.019
2026-09-15 19:36:17,205 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:18,371 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.852 ± 0.034
2026-09-15 19:36:18,384 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:20,712 | IN

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std,total_rows,%_rows_deleted,total_rows x mean,target_mean
threshold,,,,,,,,,,,
7,0.817073,0.857143,0.825000,0.853503,0.867470,0.844038,0.019493,727,1.222826,613.615494,0.551582
6,0.816568,0.823529,0.835443,0.836364,0.849673,0.832315,0.011425,736,0.000000,612.584184,0.552989
8,0.800000,0.872483,0.887417,0.825000,0.877419,0.852464,0.033918,690,6.250000,588.200132,0.546377
9,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627,688,6.521739,586.914157,0.545058
10,0.870130,0.829630,0.828571,0.861314,0.819444,0.841818,0.020031,649,11.820652,546.339784,0.546995
11,0.850000,0.877193,0.904348,0.873950,0.833333,0.867765,0.024358,474,35.597826,411.320489,0.613924
12,0.869565,0.857143,0.868687,0.888889,0.844444,0.865746,0.014742,401,45.516304,347.164008,0.586035
13,0.808511,0.800000,0.840000,0.830189,0.875000,0.830740,0.026397,262,64.402174,217.653844,0.473282


Based on the results of testing the multiple threshold above, a threshold of twelve, which retains only rows with at least twelve non-missing features, results in the highest F1-score. However, this threshold has tow major drawbacks: it discards thirty-six percent of the training data and results in a substantial difference in the target mean compared with the original dataset.
Three threshold satisfy our previously defined constraints: seven, eight, and nine while six represent the original dataset. Amongthese feasible threshold, we choose a threshold of nine because it achieves a hogher F1-score while stll satisfying both constrains.

In [6]:
len(X_train.columns)

13

The threshold selection process is based on `X_train` with `y_train` and depends on the total number of features. Therefore, the threshold values should be adjusted according to the dimensionality of the dataset.


In [7]:
mask = X_train.notna().sum(axis=1) >= 8

X_train = X_train.loc[mask]
y_train = y_train.loc[mask]

In [8]:
X_train.shape

(688, 13)

In [9]:
strategies = {
    "indicators_both": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=True),
        categorical_pipeline=create_categorical_pipeline(add_indicator=True),
    ),
    "indicators_numeric": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=True),
        categorical_pipeline=create_categorical_pipeline(add_indicator=False),
    ),
    "indicators_categorical": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=True),
    ),
    "indicators_none": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=False),
    ),
}

In [10]:
list_of_scores = []

for name, strategy in strategies.items():
    pipeline = create_training_pipeline(
        model=LogisticRegression(random_state=RANDOM_STATE), preprocessor=strategy
    )

    scores = t.cross_validate_model(
        pipeline, X_train, y_train, t.TrainingConfig(ExperimentConfig())
    )

    scores["strategy"] = name
    list_of_scores.append(scores)

list_of_scores = (
    pd.concat(list_of_scores).sort_values("mean", ascending=False).set_index("strategy")
)
list_of_scores

2026-09-15 19:36:24,785 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:26,156 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.853 ± 0.024
2026-09-15 19:36:26,162 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:26,958 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.839 ± 0.018
2026-09-15 19:36:26,961 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:36:28,267 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.858 ± 0.021
2026-09-15 19:36:28,280 | IN

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
strategy,,,,,,,
indicators_categorical,0.845638,0.860759,0.826667,0.887324,0.870130,0.858104,0.020733
indicators_both,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627
indicators_numeric,0.832215,0.842767,0.807947,0.853147,0.860927,0.839401,0.018464
indicators_none,0.826667,0.828025,0.800000,0.834532,0.853503,0.828546,0.017189


In [11]:
selected_preprocessor = strategies["indicators_categorical"]

# Evaluate Different Algorithm 

In [12]:
list_of_models = [
    LogisticRegression(random_state=42),
    RandomForestClassifier(random_state=42),
    CalibratedClassifierCV(SVC(), ensemble=False),
    GradientBoostingClassifier(random_state=42),
    KNeighborsClassifier(),
    XGBClassifier(random_state=42),
]

In [13]:
start_experiment()

experiment_config = ExperimentConfig()

training_config = t.TrainingConfig(experiment=experiment_config)

t.model_comparison_cv(
    list_of_models,
    create_training_pipeline,
    X_train,
    y_train,
    training_config,
)

2026-09-15 19:36:32,608 | INFO | heart_disease.models.train | Starting cross-validation for LogisticRegression
2026-09-15 19:37:51,238 | INFO | heart_disease.models.train | Finished LogisticRegression: F1 = 0.8531 ± 0.0236
2026-09-15 19:37:51,244 | INFO | heart_disease.models.train | Starting cross-validation for RandomForestClassifier


🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/7d90ab2c69314b4a82b4bae24acd8efd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026-09-15 19:37:56,145 | INFO | heart_disease.models.train | Finished RandomForestClassifier: F1 = 0.8406 ± 0.0326
2026-09-15 19:37:56,147 | INFO | heart_disease.models.train | Starting cross-validation for CalibratedClassifierCV


🏃 View run RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/eebc5dca1f0943b4b4c80428246caf15
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026-09-15 19:37:58,340 | INFO | heart_disease.models.train | Finished CalibratedClassifierCV: F1 = 0.8502 ± 0.0217
2026-09-15 19:37:58,341 | INFO | heart_disease.models.train | Starting cross-validation for GradientBoostingClassifier


...

2026-09-15 19:38:03,114 | INFO | heart_disease.models.train | Finished KNeighborsClassifier: F1 = 0.8330 ± 0.0189
2026-09-15 19:38:03,118 | INFO | heart_disease.models.train | Starting cross-validation for XGBClassifier


🏃 View run KNeighborsClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/bd1493a2bbc64b3e92edf8fcecb58473
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026-09-15 19:38:06,090 | INFO | heart_disease.models.train | Finished XGBClassifier: F1 = 0.8378 ± 0.0179


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/daea41ad9d3840cd88aec0e7d6f76482
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,Model,fit_time,Accuracy,Precision,Recall,F1,ROC AUC
0,LogisticRegression,0.4764,0.8402,0.8543,0.8533,0.8531,0.9106
1,CalibratedClassifierCV,0.5100,0.8329,0.8310,0.8720,0.8502,0.9026
2,GradientBoostingClassifier,1.3450,0.8256,0.8359,0.8480,0.8415,0.8855
3,RandomForestClassifier,1.4239,0.8257,0.8404,0.8427,0.8406,0.8879
4,XGBClassifier,0.5295,0.8198,0.8211,0.8560,0.8378,0.8727
5,KNeighborsClassifier,0.1706,0.8169,0.8294,0.8400,0.8330,0.8690


In [14]:
from mlflow import search_runs

search_runs(
    experiment_names=["uci-heart-disease"]
)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.f1,metrics.fit_time,metrics.accuracy,metrics.roc_auc,metrics.recall,metrics.precision,params.numeric_missing_indicator,params.enable_categorical,params.objective,params.categorical_missing_indicator,params.random_state,params.cv_folds,params.test_size,params.model_name,params.drop_threshold,params.multiple_scoring,params.missing,params.single_scoring,params.use_scaler,params.metric,params.p,params.weights,params.n_neighbors,params.algorithm,params.leaf_size,params.min_samples_leaf,params.validation_fraction,params.criterion,params.tol,params.min_impurity_decrease,params.learning_rate,params.subsample,params.verbose,params.min_weight_fraction_leaf,...,params.tree_method,params.min_child_weight,params.booster,params.gamma,params.importance_type,params.num_parallel_tree,params.eval_metric,params.max_leaves,params.interaction_constraints,params.n_jobs,params.callbacks,params.reg_lambda,params.max_cat_threshold,params.device,params.colsample_bynode,params.colsample_bytree,params.monotone_constraints,params.max_delta_step,params.scale_pos_weight,params.base_score,params.validate_parameters,params.multi_strategy,params.feature_weights,params.max_cat_to_onehot,params.metric_params,params.max_leaf_nodes,params.n_iter_no_change,params.init,params.estimator__random_state,params.estimator__class_weight,params.cv,params.estimator,params.max_samples,params.class_weight,params.monotonic_cst,params.logistic_regression_params,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.runName
0,daea41ad9d3840cd88aec0e7d6f76482,1,FINISHED,mlflow-artifacts:/1/daea41ad9d3840cd88aec0e7d6...,2026-09-15 12:38:04.935000+00:00,2026-09-15 12:38:05.650000+00:00,0.837846,0.529550,0.819782,0.872737,0.856000,0.821052,False,True,binary:logistic,True,42,5,0.2,XGBClassifier,9,"['accuracy', 'precision', 'recall', 'f1', 'roc...",nan,f1,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,irasionize,03_data_preprocessing_and_model_training.ipynb,NOTEBOOK,XGBClassifier
1,bd1493a2bbc64b3e92edf8fcecb58473,1,FINISHED,mlflow-artifacts:/1/bd1493a2bbc64b3e92edf8fcec...,2026-09-15 12:38:02.395000+00:00,2026-09-15 12:38:02.924000+00:00,0.832956,0.170586,0.816905,0.869011,0.840000,0.829369,False,None,None,True,42,5,0.2,KNeighborsClassifier,9,"['accuracy', 'precision', 'recall', 'f1', 'roc...",None,f1,True,minkowski,2,uniform,5,auto,30,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,irasionize,03_data_preprocessing_and_model_training.ipynb,NOTEBOOK,KNeighborsClassifier
2,4bf0eda8cebd4192a0b3c60e0abb1629,1,FINISHED,mlflow-artifacts:/1/4bf0eda8cebd4192a0b3c60e0a...,2026-09-15 12:38:00.751000+00:00,2026-09-15 12:38:01.248000+00:00,0.841470,1.344993,0.825621,0.885511,0.848000,0.835921,False,None,None,True,42,5,0.2,GradientBoostingClassifier,9,"['accuracy', 'precision', 'recall', 'f1', 'roc...",None,f1,True,None,None,None,None,None,None,1,0.1,deprecated,0.0001,0.0,0.1,1.0,0,0.0,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,irasionize,03_data_preprocessing_and_model_training.ipynb,NOTEBOOK,GradientBoostingClassifier
3,06b6812fde6a44539ec644e9c1093c30,1,FINISHED,mlflow-artifacts:/1/06b6812fde6a44539ec644e9c1...,2026-09-15 12:37:57.719000+00:00,2026-09-15 12:37:58.170000+00:00,0.850214,0.510021,0.832889,0.902626,0.872000,0.831030,False,None,None,True,42,5,0.2,CalibratedClassifierCV,9,"['accuracy', 'precision', 'recall', 'f1', 'roc...",None,f1,True,None,None,None,None,None,Non

# Tune Hyperparameters for Selected Algorithm

In [15]:
param_grid = [
    {
        "classifier__max_iter": [5000],
        "classifier__solver": ["lbfgs"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "classifier__max_iter": [5000],
        "classifier__solver": ["saga"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__l1_ratio": [0, 0.5, 1],
    },
]

In [16]:
selected_pipeline = create_training_pipeline(
    LogisticRegression(random_state=RANDOM_STATE), selected_preprocessor
)
search = t.grid_search(
    selected_pipeline,
    param_grid,
    X_train,
    y_train,
    t.TrainingConfig(ExperimentConfig()),
)

2026-09-15 19:38:09,035 | INFO | heart_disease.models.train | Starting grid search


2026-09-15 19:38:28,866 | INFO | heart_disease.models.train | Best parameters: {'classifier__C': 1, 'classifier__l1_ratio': 0.5, 'classifier__max_iter': 5000, 'classifier__solver': 'saga'} | Best CV scores: 0.861


In [17]:
search.best_score_

np.float64(0.861117598834755)

In [18]:
selected_model = create_training_pipeline(
    LogisticRegression(
        max_iter=5000, l1_ratio=0.5, C=1, solver="saga", random_state=RANDOM_STATE
    ),
    selected_preprocessor,
)


t.cross_validate_model(
    selected_model, X_train, y_train, t.TrainingConfig(ExperimentConfig())
)

2026-09-15 19:38:28,957 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:38:30,669 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.861 ± 0.024


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.845638,0.860759,0.826667,0.895105,0.877419,0.861118,0.023864


In [19]:
base_model = create_training_pipeline(LogisticRegression(random_state=RANDOM_STATE))


t.cross_validate_model(
    base_model, X_train, y_train, t.TrainingConfig(ExperimentConfig())
)

2026-09-15 19:38:30,740 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-15 19:38:31,674 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.853 ± 0.024


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627


In [20]:
selected_model = t.train_model(selected_model, X_train, y_train)

2026-09-15 19:38:32,238 | INFO | heart_disease.models.train | Training: LogisticRegression with shape of dataframe: (688, 13)


In [21]:
path = MODEL_DIR / "01_selected_model_LR.joblib"
t.save_model(selected_model, path)

2026-09-15 19:38:32,280 | INFO | heart_disease.models.train | Saving LogisticRegression model to /home/irasionize/heart-disease-ml/models/01_selected_model_LR.joblib
